# Triage Acuity Prediction Pipeline
This notebook demonstrates a machine learning pipeline using LightGBM to predict patient triage levels (ESI) from clinical data and chief complaints.

This notebook was made for a kaggle hackathon called Triagegeist.

# Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import shap
SEED = 2025
np.random.seed(SEED)

# Paths — update if running locally
DATA_PATH = '/kaggle/input/triagegeist/'

print('Libraries loaded successfully.')
print(f'LightGBM version: {lgb.__version__}')

# Load Data

In [ ]:
train=pd.read_csv('/kaggle/input/competitions/triagegeist/train.csv')
test=pd.read_csv('/kaggle/input/competitions/triagegeist/test.csv')
cc=pd.read_csv('/kaggle/input/competitions/triagegeist/chief_complaints.csv')
history=pd.read_csv('/kaggle/input/competitions/triagegeist/patient_history.csv')
print(f'Train:            {train.shape[0]:,} rows x {train.shape[1]} cols')
print(f'Test:             {test.shape[0]:,} rows x {test.shape[1]} cols')
print(f'Chief complaints: {cc.shape[0]:,} rows x {cc.shape[1]} cols')
print(f'Patient history:  {history.shape[0]:,} rows x {history.shape[1]} cols')
train.head(3)

# Data Analysis

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

acu_counts = train['triage_acuity'].value_counts().sort_index()
acu_labels = ['ESI-1\n(Immediate)', 'ESI-2\n(Emergent)', 'ESI-3\n(Urgent)',
              'ESI-4\n(Less Urgent)', 'ESI-5\n(Non-Urgent)']
colors = ['#d32f2f','#f57c00','#fbc02d','#388e3c','#1976d2']

axes[0].bar(acu_labels, acu_counts.values, color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('Triage Acuity Distribution (Train)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Patient Count')
for i, v in enumerate(acu_counts.values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontsize=9)

# NEWS2 score by acuity
train.boxplot(column='news2_score', by='triage_acuity', ax=axes[1],
              boxprops=dict(color='steelblue'),
              medianprops=dict(color='crimson', linewidth=2))
axes[1].set_title('NEWS2 Score by Acuity Level', fontsize=13, fontweight='bold')
axes[1].set_xlabel('ESI Acuity Level')
axes[1].set_ylabel('NEWS2 Score')
axes[1].set_xticklabels(['ESI-1', 'ESI-2', 'ESI-3', 'ESI-4', 'ESI-5'])
plt.tight_layout()
plt.show()

In [ ]:
# Missing value analysis
miss = train.isnull().sum()
miss = miss[miss > 0].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 3))
miss_pct = (miss / len(train) * 100).round(1)
ax.barh(miss_pct.index, miss_pct.values, color='steelblue', edgecolor='white')
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Values in Train Set', fontsize=12, fontweight='bold')
for i, v in enumerate(miss_pct.values):
    ax.text(v + 0.1, i, f'{v}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print('\nNote: pain_score = -1 means not recorded (not a true NaN).')
print(f'pain_score -1 count: {(train["pain_score"] == -1).sum():,}')

In [ ]:
# Vital signs by acuity — heatmap of medians
vital_cols = ['systolic_bp','heart_rate','respiratory_rate','temperature_c','spo2','gcs_total','news2_score']
medians = train.groupby('triage_acuity')[vital_cols].median()

fig, ax = plt.subplots(figsize=(11, 3))
sns.heatmap(medians.T, annot=True, fmt='.1f', cmap='RdYlGn_r',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Median Value'})
ax.set_xlabel('ESI Acuity Level')
ax.set_title('Median Vital Signs by Triage Acuity', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Feature Engineering

In [ ]:
def prepare_features(df, cc_df, hist_df, tfidf=None, fit_tfidf=False):
    df = df.copy()

    # Join history and complaints
    df = df.merge(hist_df, on='patient_id', how='left')
    df = df.merge(cc_df[['patient_id','chief_complaint_raw']], on='patient_id', how='left')

    #pain_score: clinical missingness signal
    df['pain_not_recorded'] = (df['pain_score'] == -1).astype(int)
    df.loc[df['pain_score'] == -1, 'pain_score'] = np.nan
    df['pain_score'] = df.groupby('age_group')['pain_score'].transform(
        lambda x: x.fillna(x.median()))

    # Missingness indicators for BP and RR
    for col in ['systolic_bp','diastolic_bp','mean_arterial_pressure',
                'pulse_pressure','shock_index','respiratory_rate']:
        df[f'{col}_missing'] = df[col].isnull().astype(int)

    # Impute remaining numerics with age_group + shift medians
    num_cols = ['systolic_bp','diastolic_bp','mean_arterial_pressure',
                'pulse_pressure','shock_index','respiratory_rate','temperature_c']
    for col in num_cols:
        df[col] = df.groupby(['age_group','shift'])[col].transform(
            lambda x: x.fillna(x.median()))
        df[col] = df[col].fillna(df[col].median())
        # Derived features
    df['elderly']     = (df['age'] >= 65).astype(int)
    df['pediatric']   = (df['age'] < 16).astype(int)
    df['night_shift'] = (df['shift'] == 'night').astype(int)
    df['weekend']     = df['arrival_day'].isin(['Saturday','Sunday']).astype(int)
    df['high_risk_arrival'] = df['arrival_mode'].isin(['ambulance','helicopter']).astype(int)
    df['altered_ms']  = df['mental_status_triage'].isin(['confused','drowsy','unresponsive','agitated']).astype(int)

    # TF-IDF on chief complaint text
    df['chief_complaint_raw'] = df['chief_complaint_raw'].fillna('unknown')
    if fit_tfidf:
        tfidf = TfidfVectorizer(max_features=200, ngram_range=(1,2),
                                min_df=5, sublinear_tf=True)
        tfidf_matrix = tfidf.fit_transform(df['chief_complaint_raw'])
    else:
        tfidf_matrix = tfidf.transform(df['chief_complaint_raw'])

    tfidf_df = pd.DataFrame(
        tfidf_matrix.toarray(),
        columns=[f'cc_{c}' for c in tfidf.get_feature_names_out()],
        index=df.index
    )
    df = pd.concat([df.reset_index(drop=True), tfidf_df.reset_index(drop=True)], axis=1)
     # Label encode categoricals
    cat_cols = ['arrival_mode','arrival_day','arrival_season','shift','age_group',
                'sex','language','insurance_type','transport_origin',
                'pain_location','mental_status_triage','chief_complaint_system','site_id']
    for col in cat_cols:
        if col in df.columns:
            df[col] = LabelEncoder().fit_transform(df[col].astype(str))

    #Drop non-feature columns
    drop_cols = ['patient_id','triage_nurse_id','chief_complaint_raw',
                 'disposition','ed_los_hours','triage_acuity']
    feat_cols = [c for c in df.columns if c not in drop_cols]

    return df[feat_cols], tfidf

print('Feature engineering function defined.')

In [ ]:
# Prepare features
X_train, tfidf_fitted = prepare_features(train, cc, history, fit_tfidf=True)
X_test,  _            = prepare_features(test,  cc, history, tfidf=tfidf_fitted, fit_tfidf=False)
y_train = train['triage_acuity'].values - 1  # 0-indexed for LightGBM

print(f'X_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')
print(f'y_train classes: {np.unique(y_train)}')

# Training Model using LightGBM

In [ ]:
lgb_params = {
    'objective':       'multiclass',
    'num_class':       5,
    'metric':          'multi_logloss',
    'n_estimators':    800,
    'learning_rate':   0.05,
    'num_leaves':      63,
    'max_depth':       -1,
    'min_child_samples': 30,
    'subsample':       0.85,
    'colsample_bytree':0.85,
    'reg_alpha':       0.1,
    'reg_lambda':      0.1,
    'class_weight':    'balanced',
    'random_state':    SEED,
    'verbose':         -1,
    'n_jobs':          -1,
}

N_FOLDS   = 5
skf       = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_preds = np.zeros((len(X_train), 5))
test_preds= np.zeros((len(X_test),  5))
qwk_scores= []
models    = []
for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train, y_train)):
    print(f'\nFold {fold+1}/{N_FOLDS}')
    X_tr, X_vl = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_vl = y_train[train_idx], y_train[valid_idx]
    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dvalid = lgb.Dataset(X_vl, label=y_vl)
    callbacks = [lgb.log_evaluation(period=100), lgb.early_stopping(stopping_rounds=100, verbose=False)]
    model  = lgb.train(lgb_params, dtrain, valid_sets=[dtrain, dvalid],
                       callbacks=callbacks)
    models.append(model)
    oof_preds[valid_idx] = model.predict(X_vl)
    test_preds += model.predict(X_test) / N_FOLDS
    qwk_scores.append(cohen_kappa_score(y_vl, np.argmax(oof_preds[valid_idx], axis=1), weights='quadratic'))
    print(f'QWK: {qwk_scores[-1]:.4f}')

print(f'\nAverage QWK: {np.mean(qwk_scores):.4f}')
print(f'Out of folds QWK: {cohen_kappa_score(y_train, np.argmax(oof_preds, axis=1), weights="quadratic"):.4f}')

# Evaluation

In [ ]:
#Classification report
target_names = ['ESI-1','ESI-2','ESI-3','ESI-4','ESI-5']
oof_labels = np.argmax(oof_preds, axis=1)
print('Classification Report (OOF):')
print(classification_report(y_train, oof_labels, target_names=target_names))

In [ ]:
#Confusion matrix
cm = confusion_matrix(y_train, oof_labels)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Reds',
            xticklabels=target_names, yticklabels=target_names,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Row %'})
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title('Confusion Matrix — OOF Predictions (row-normalised %)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

#Feature Importance

In [ ]:
# Aggregate feature importance across folds
feat_imp = pd.DataFrame({
    'feature':    X_train.columns,
    'importance': np.mean([m.feature_importance(importance_type='gain') for m in models], axis=0)
}).sort_values('importance', ascending=False)

top30 = feat_imp.head(30)

fig, ax = plt.subplots(figsize=(10, 8))
# Mapping colors: Structured vs NLP features
colors_imp = ['purple' if not c.startswith('cc_') else '#1976d2'
              for c in top30['feature']]

ax.barh(top30['feature'][::-1], top30['importance'][::-1],
        color=colors_imp[::-1], edgecolor='white')
ax.set_xlabel('Mean Feature Importance (gain)')
ax.set_title('Top 30 Features — Purple: Structured  |  Blue: NLP (TF-IDF)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

# Final Submission

In [ ]:
# Final Submission Generation
submission = pd.DataFrame({
    'patient_id': test['patient_id'],
    'triage_acuity': np.argmax(test_preds, axis=1) + 1  # Convert 0-indexed back to 1-5
})

submission.to_csv('submission.csv', index=False)
print(f'Submission file created with {len(submission)} rows.')
print('\nFirst 5 rows of submission:')
print(submission.head())